In [10]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import urllib.parse
import time
import random

In [ ]:
df = pd.read_csv("../data/sample100.csv")
df.columns = df.columns.str.strip()
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
]
session = requests.Session()
VAT_PATTERN = r"(?:GB|VAT)?\s*([0-9]{3}\s?[0-9]{4}\s?[0-9]{2}|[0-9]{9})\b"

print(f"Sample loaded: {len(df)} companies")
df.head(3)

Sample loaded: 100 companies


,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,HOCKEY STOP LTD,12343252,NaN,NaN,UNIT 19,ST. HILARY PARK ROAD,KING'S LYNN,NORFOLK,ENGLAND,PE30 4ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15/12/2026,01/12/2025
1,OAKSIDE VENTURES LTD,15802979,NaN,NaN,OFFICE 7,35-37 LUDGATE HILL,LONDON,NaN,UNITED KINGDOM,EC4M 7JN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/07/2027,24/06/2026
2,SPIRAL CONSULTANTS LIMITED,08905847,NaN,NaN,WOODEND,HAMSTREET ROAD,ASHFORD,KENT,UNITED KINGDOM,TN26 2EB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,07/03/2027,21/02/2026


In [12]:
def clean_company_name(name: str) -> str:
    cleaned = re.sub(r"\(.*?\)", "", str(name))
    cleaned = re.sub(
        r"\b(LIMITED|LTD|PLC|LLP|INC|CORP|HOLDINGS|GROUP|\.COM)\b",
        "", cleaned, flags=re.IGNORECASE
    )
    cleaned = re.sub(r"[^\w\s]", " ", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()

In [ ]:
def check_uk_mod97(vat_str: str) -> bool:
    clean = re.sub(r"\D", "", str(vat_str))
    if len(clean) != 9 or clean.startswith(("07", "08", "01", "02", "03", "00")):
        return False
    digits = [int(d) for d in clean]
    weights = [8, 7, 6, 5, 4, 3, 2]
    total = sum(d * w for d, w in zip(digits[:7], weights))
    check_val = (digits[7] * 10) + digits[8]
    return ((total + check_val) % 97 == 0) or ((total + check_val + 55) % 97 == 0)

In [ ]:
def search_discovery_multi(company_name: str, company_number: str = "", postcode: str = ""):
  
    candidates = []
    clean_n = clean_company_name(company_name)
    headers = {"User-Agent": random.choice(user_agents), "Accept-Language": "en-GB,en;q=0.9"}

    # Yahoo
    try:
        q = f'"{clean_n}" VAT OR "{clean_n}" "GB"'
        r = requests.get("https://search.yahoo.com/search", params={"p": q, "n": 10}, headers=headers, timeout=4)
        if r.status_code == 200:
            for m in re.findall(VAT_PATTERN, r.text, flags=re.IGNORECASE):
                d = re.sub(r"\D", "", m)
                if check_uk_mod97(d):
                    candidates.append(d)
    except Exception:
        pass

    # DuckDuckGo Lite 
    if not candidates:
        try:
            r = requests.post("https://lite.duckduckgo.com/lite/", data={"q": f"{clean_n} VAT number UK"}, headers=headers, timeout=4)
            if r.status_code == 200:
                for m in re.findall(VAT_PATTERN, r.text, flags=re.IGNORECASE):
                    d = re.sub(r"\D", "", m)
                    if check_uk_mod97(d):
                        candidates.append(d)
        except Exception:
            pass

    # Bing HTML 
    if not candidates:
        try:
            r = requests.get(f"https://www.bing.com/search?q={urllib.parse.quote(clean_n + ' VAT UK')}", headers=headers, timeout=4)
            if r.status_code == 200:
                for m in re.findall(VAT_PATTERN, r.text, flags=re.IGNORECASE):
                    d = re.sub(r"\D", "", m)
                    if check_uk_mod97(d):
                        candidates.append(d)
        except Exception:
            pass

    return list(set(candidates))

In [ ]:
test_candidates = search_discovery_multi("TESCO PLC", "00445790", "")
print("Test candidates for TESCO PLC:", test_candidates)

Test candidates for TESCO PLC: ['220430231']


In [ ]:
import requests
from bs4 import BeautifulSoup

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
})

url = "https://www.tax.service.gov.uk/check-vat-number/enter-vat-details"
r_get = session.get(url, timeout=10)
print("GET status:", r_get.status_code)
print("Cookies:", session.cookies.get_dict())

if r_get.status_code != 200:
    print(r_get.text[:500])
    raise SystemExit

soup = BeautifulSoup(r_get.text, "html.parser")
form = soup.find("form")
if form:
    print("action:", form.get("action"))
    print("method:", form.get("method"))
else:
    print("No <form> found in static HTML.")

inputs = soup.find_all("input")
for inp in inputs:
    print(f"  name={inp.get('name')!r}  type={inp.get('type')!r}  value={inp.get('value')!r}")

payload = {}
for inp in inputs:
    name = inp.get("name")
    if name:
        payload[name] = inp.get("value", "")
payload["target"] = "220430231"
payload.pop("withConsultationNumber", None)

r_post = session.post(url, data=payload, timeout=10)
print("POST status:", r_post.status_code)
print("POST final URL:", r_post.url)
print(r_post.text[:1000])
text_upper = r_post.text.upper()
for marker in ["VALID UK VAT NUMBER", "VAT REGISTRATION DETAILS", "INVALID"]:
    print(f"{marker!r} present: {marker in text_upper}")

GET status: 200
Cookies: {'mdtpdi': 'mdtpdi#1272e850-7b0b-47d6-a7cd-7d8e730e949f#1786983035478_WUXz57hmZkMkYjr1rg1jKA==', 'mdtp': 'K3j9PG+22iDM5NfdXNNua+vUhGCfIpkgse03GPXZzY4ogpx4J9dp/xUwqYy4ejo0X77T6bD7MvCnGtYgEcL1DZOXkkbCuMXVzuI6gE0uSM8ZJEEwd9Kl4r1r1rFvgW5Ehg4vupsNEUhmuG+DmuAHm3oyEpHrYtov+zWWMq6xa2fJnLXHodYZzbbERIFAebn82lHAgzWtd1f6+HyZzMqVgt3WG7N6vIFiCG1J3tCHpwMXxe/BisOHiI+JpTOqj6rHCQ9LEVBuyKaCqWaT3zgiR251IbkKLUY+PpiCO9bYkCSAsvbYzudW/PAd'}
action: /check-vat-number/enter-vat-details
method: POST
  name='csrfToken'  type='hidden'  value='d4e41324cac9c611ece09514b6e18f4c324382fb-1786983035479-6d477343ebf757c0ddf4df69'
  name='target'  type='text'  value=None
  name='withConsultationNumber'  type='checkbox'  value='true'
  name='requester'  type='text'  value=None
POST status: 200
POST final URL: https://www.tax.service.gov.uk/check-vat-number/known





















<!DOCTYPE html>
<html lang="en" class="govuk-template ">
  <head>
    <meta charset="utf-8">
    <title>Valid UK VAT n

In [ ]:
def check_vat_govuk_or(vat_number: str, row_data) -> tuple:
    """Validate a VAT candidate against the official HMRC checker and confirm it
    belongs to the right company using name/postcode/town/address signals."""
    clean_vat = "".join(filter(str.isdigit, str(vat_number)))
    if len(clean_vat) != 9:
        return False, "Invalid_Length"
    url = "https://www.tax.service.gov.uk/check-vat-number/enter-vat-details"
    try:
        session = requests.Session()  # fresh session per call, not shared across the whole loop
        session.headers.update({"User-Agent": random.choice(user_agents)})
        r_get = session.get(url, timeout=5)
        soup_get = BeautifulSoup(r_get.text, "html.parser")
        csrf_input = soup_get.find("input", {"name": "csrfToken"})
        csrf_token = csrf_input.get("value", "") if csrf_input else ""

        payload = {
            "csrfToken": csrf_token,
            "target": clean_vat,   # confirmed field name from the live HMRC form
        }
        r_post = session.post(url, data=payload, timeout=5)
        page_text = r_post.text.upper()

        if "VALID UK VAT NUMBER" not in page_text and "VAT REGISTRATION DETAILS" not in page_text:
            return False, "Invalid_HMRC"

        # company name 
        clean_name = clean_company_name(str(row_data.get("CompanyName", "")))
        name_tokens = [t for t in re.split(r"[^\w]", clean_name.upper()) if len(t) >= 4]
        for tok in name_tokens:
            if tok in page_text:
                return True, f"Match_Name ({tok})"

        #  postcode 
        postcode = str(row_data.get("RegAddress.PostCode", "")).upper().strip()
        if postcode:
            pc_no_spaces = postcode.replace(" ", "")
            page_no_spaces = page_text.replace(" ", "")
            if postcode in page_text or pc_no_spaces in page_no_spaces:
                return True, f"Match_PostCode ({postcode})"

        #  town 
        town = str(row_data.get("RegAddress.PostTown", "")).upper().strip()
        if town and len(town) >= 4 and town in page_text:
            return True, f"Match_Town ({town})"

        # street/address 
        addr1 = str(row_data.get("RegAddress.AddressLine1", "")).upper().strip()
        addr_tokens = [
            t for t in re.split(r"[^\w]", addr1)
            if len(t) >= 4 and t not in ["ROAD", "STREET", "AVENUE", "HOUSE", "LANE", "UNIT"]
        ]
        for t in addr_tokens:
            if t in page_text:
                return True, f"Match_Address ({t})"

        return False, "Different_Company"
    except Exception as e:
        return False, f"Error_{str(e)}"


In [ ]:

raw_candidates_list = []
verified_vats = []
validation_statuses = []

for idx, row in df.iterrows():
    c_name = str(row.get("CompanyName", "")).strip()
    c_num = str(row.get("CompanyNumber", "")).strip()
    c_postcode = str(row.get("RegAddress.PostCode", "")).strip()

    print(f"[{idx+1}/{len(df)}] {c_name}...", end="", flush=True)
    candidates = search_discovery_multi(c_name, c_num, c_postcode)
    raw_candidates_list.append(", ".join(candidates) if candidates else None)
    matched_vat = None
    status = "No_VAT_Found"

    if candidates:
        status = "Invalid_HMRC"
        for cand in candidates:
            is_match, reason = check_vat_govuk_or(cand, row)
            time.sleep(0.4)
            if is_match:
                matched_vat = cand
                status = "Valid_HMRC_Matched"
                print(f" ===> [CONFIRMED: {reason}] VAT: {cand}", end="", flush=True)
                break
            elif reason == "Different_Company":
                status = "Valid_HMRC_Different_Company"

    if not matched_vat and not candidates:
        print(" -> [No data]", flush=True)
    elif not matched_vat:
        print(f" -> [Candidates: {candidates} | Status: {status}]", flush=True)
    else:
        print("", flush=True)
    verified_vats.append(matched_vat)
    validation_statuses.append(status)
    time.sleep(1.0)

=== STARTING FULL PIPELINE RUN ===
[1/100] HOCKEY STOP LTD... ===> [CONFIRMED: Match_Name (HOCKEY)] VAT: 360575690
[2/100] OAKSIDE VENTURES LTD... -> [Candidates: ['922579842'] | Status: Valid_HMRC_Different_Company]
[3/100] SPIRAL CONSULTANTS LIMITED... ===> [CONFIRMED: Match_Name (SPIRAL)] VAT: 314546025
[4/100] UTOP SERVICES LTD... -> [No data]
[5/100] THE RECRUITMENT CONTACT LTD... ===> [CONFIRMED: Match_Name (RECRUITMENT)] VAT: 363441310
[6/100] HEALTHWISE SUPPLIES LTD... ===> [CONFIRMED: Match_Name (HEALTHWISE)] VAT: 799342081
[7/100] QDM MORTGAGES LTD... -> [No data]
[8/100] THE ZOO HOLBROOK LTD... -> [No data]
[9/100] GLW BUILDING LTD... ===> [CONFIRMED: Match_Name (BUILDING)] VAT: 411177922
[10/100] KODIVA LTD... -> [Candidates: ['200462227'] | Status: Valid_HMRC_Different_Company]
[11/100] VELEZZA YACHT CHARTERS LIMITED... ===> [CONFIRMED: Match_Name (VELEZZA)] VAT: 503774495
[12/100] DOWN TOWN (AS) LIMITED... ===> [CONFIRMED: Match_Name (DOWN)] VAT: 469018570
[13/100] BERNAR

In [ ]:
df["Raw_Extracted_VAT"] = raw_candidates_list
df["Verified_VAT"] = verified_vats
df["Validation_Status"] = validation_statuses
df.to_csv("../outputs/sample_final_evaluated.csv", index=False)
total_extracted = (df["Validation_Status"] != "No_VAT_Found").sum()
true_positives = (df["Validation_Status"] == "Valid_HMRC_Matched").sum()
false_positives = total_extracted - true_positives
fpr = (false_positives / total_extracted * 100) if total_extracted > 0 else 0
print(f"Total companies in sample: {len(df)}")
print(f"Companies with at least one candidate extracted: {total_extracted}")
print(f"True Positives (confirmed by HMRC + correct-company match): {true_positives}")
print(f"False Positives (checksum-valid, wrong company): {false_positives}")
print(f"False Positive Rate (FPR): {fpr:.2f}%")


Total companies in sample: 100
Companies with at least one candidate extracted: 57
True Positives (confirmed by HMRC + correct-company match): 27
False Positives (checksum-valid, wrong company): 30
False Positive Rate (FPR): 52.63%
